In [ ]:
#| hide
from ossl_fuku_transfer.data import *

# OSSL to Fukushima transfer

> Test whether OSSL-trained MIR models help predict Fukushima soil properties

::: {.callout-warning title="Work in progress"}
The findings change as new experiments run and as the Fukushima data team answers the questions below.
:::

This project tests whether CNN models of soil mid-infrared (MIR) spectra, trained on the [Open Soil Spectral Library (OSSL)](https://explorer.soilspectroscopy.org/), help predict soil properties from spectra of Fukushima soils. OSSL was measured on Bruker instruments and the Fukushima soils on a Thermo Nicolet. The targets are exchangeable potassium (Kex), cation exchange capacity (CEC) and the ratio of exchangeable to total Cs-137. The Fukushima data are 1,389 soil samples from 391 fields, sampled from 2013 to 2020.

The audience is soil scientists working with the Fukushima data and anyone transferring spectral models between instruments. Each notebook asks one question. The table under "Notebooks" gives each question and its answer. The questions under "Questions for the Fukushima data team" need answers from the people who sampled and measured the soils.

## Findings

OSSL models fail on the Fukushima spectra without retraining. Without fine-tuning, the OSSL Kex model gives R² below 0 and a correlation of 0.13 on the Fukushima test samples. The OSSL CEC model gives a correlation of 0.57, with a large bias.

Trained on Fukushima labels, OSSL helps Kex a little on fields already seen in training, and not at all on new fields. On the Cs137 study's test samples, full fine-tuning gives Kex R² 0.41 against 0.37 for a CNN trained on Fukushima only. For CEC it gives 0.74 against 0.71. When whole fields are held out, ridge regression on frozen OSSL features gives Kex R² 0.26 against 0.28 for ridge regression on the spectra.

For the Cs-137 ratio, frozen OSSL features help on new fields, but only slightly. All three models below use ridge regression, scored with folds grouped by field; only the input changes. Ridge on the preprocessed spectra (SNV, then a Savitzky-Golay first derivative) gives R² 0.52. Ridge on the activations of block 4 of the two CNNs trained on OSSL to predict Kex and CEC, with their weights frozen, gives R² 0.57. Ridge on block 4 of the same CNNs with random weights gives R² 0.52, the same as the spectra. The gain of 0.05 (bootstrap interval over fields 0.017 to 0.077) therefore comes from what the networks learned on OSSL, not from the network architecture. It cuts the unexplained variance from 48% to 43%.

Scores on the Cs137 study's split are optimistic. 89% of its 398 test samples come from a field that also has samples in training. Holding out whole fields lowers Kex R² by 0.10 to 0.13 and the Cs-137 ratio by 0.05 to 0.08.

The OSSL soils cover the Fukushima soils in property space. The OSSL Kex model predicts the closest OSSL soil to each Fukushima soil at R² 0.60. Removing the Bruker-Thermo difference from the spectra, and augmenting the spectra during pretraining, both left its Kex correlation on the Fukushima spectra near 0. Water vapour and sample moisture do not explain the failure.

In Fukushima's Lowland soils, the sharp clay-mineral bands near 3620 and 3695 cm⁻¹ relate to clay in the opposite direction to OSSL. Band depth is measured in each spectrum as minus the minimum of its second derivative near the band centre. In the Lowland soils, deeper bands go with less clay (Spearman -0.31 and -0.35, 631 samples). In the OSSL soils with the closest lab properties, deeper bands go with more clay (0.28 and 0.39). The OSSL Kex model relies most on this region, so this is one sign that OSSL's spectral relations do not hold in Fukushima soils. It has not been shown to cause the Kex failure.

On new fields, Kex can hardly be predicted within paddy fields. Ridge regression on the spectra gives R² -0.03 in paddy fields and 0.20 in upland fields. Paddy fields hold 880 of the 1,388 samples with Kex.

## Notebooks

### Data and spectra

| Notebook | Question | Answer |
|---|---|---|
| [`00_data`](https://franckalbinet.github.io/ossl-fuku-transfer/data.html) | Which samples and spectra does the project use? | The 1,389 samples of the Cs137 ratio study that have `.spa` files, after leaving out 12 samples with doubtful scans. |
| [`01_pca`](https://franckalbinet.github.io/ossl-fuku-transfer/pca.html) | How far are the Fukushima spectra from OSSL? | With SNV only, 22% of the Fukushima samples are beyond OSSL's 99th percentile of T². The first derivative removes most of this offset. The largest remaining residual is a broad O-H absorption at 3500 to 3950 cm⁻¹. |

### Kex and CEC

| Notebook | Question | Answer |
|---|---|---|
| [`02_transfer`](https://franckalbinet.github.io/ossl-fuku-transfer/transfer.html) | How are the transfer experiments run? | The functions that pretrain the CNN on OSSL, train the Fukushima models from it and score them. |
| [`03_kex`](https://franckalbinet.github.io/ossl-fuku-transfer/kex.html) | Does an OSSL Kex model help predict Fukushima Kex? | Without fine-tuning, R² is below 0 and r is 0.13. Full fine-tuning gives R² 0.41, against 0.37 for a CNN trained on Fukushima only. |
| [`04_cec`](https://franckalbinet.github.io/ossl-fuku-transfer/cec.html) | Does an OSSL CEC model help predict Fukushima CEC? | Without fine-tuning, r is 0.57 with a large bias. Full fine-tuning gives R² 0.74, against 0.71 for a CNN trained on Fukushima only. |
| [`08_learning_curve`](https://franckalbinet.github.io/ossl-fuku-transfer/learning_curve.html) | How many Fukushima samples do the Kex models need? | OSSL helps most with few samples. With 400, ridge regression on frozen OSSL features gives R² 0.33, against about 0.24 for the other models. No curve has levelled off at 1,047 samples. |
| [`09_block4`](https://franckalbinet.github.io/ossl-fuku-transfer/block4.html) | Does transferring the network only up to block 4 work better? | No. No fine-tuned starting point beats ridge regression on frozen block 4, at R² 0.43. |

### Inside the CNN

| Notebook | Question | Answer |
|---|---|---|
| [`06_dynamics`](https://franckalbinet.github.io/ossl-fuku-transfer/dynamics.html) | Does the CNN train without optimisation faults? | Yes, on OSSL and on Fukushima. Block 5 becomes sparse during training. |
| [`07_inside_transfer`](https://franckalbinet.github.io/ossl-fuku-transfer/inside_transfer.html) | How do the layers of the OSSL and Fukushima Kex models compare? | On Fukushima, every layer holds about the same Kex information, even with random weights. Block 5 and the global pool are the most OSSL-specific part of the network. |
| [`17_importance`](https://franckalbinet.github.io/ossl-fuku-transfer/importance.html) | Which wavenumbers drive the Kex predictions? | On OSSL, the OSSL Kex model relies most on the Al-OH band near 3620 cm⁻¹. On Fukushima spectra it gives that region half the weight. |

### Why the transfer is limited

| Notebook | Question | Answer |
|---|---|---|
| [`05_paired`](https://franckalbinet.github.io/ossl-fuku-transfer/paired.html) | Do Bruker and Thermo spectra of the same soils agree? | No. The Thermo spectra have about six times less contrast, and the two instruments disagree most at 650 to 1300 cm⁻¹. The OSSL Kex model fails on both. |
| [`12_wetchem`](https://franckalbinet.github.io/ossl-fuku-transfer/wetchem.html) | Are the Fukushima soils within the range of OSSL soils? | Yes. The OSSL Kex model predicts the closest OSSL soil to each Fukushima soil at R² 0.60, and the Fukushima spectra at -0.76. |
| [`13_epo`](https://franckalbinet.github.io/ossl-fuku-transfer/epo.html) | Does removing the Bruker-Thermo difference from the spectra help? | No. External parameter orthogonalisation leaves the Kex correlation near 0 without fine-tuning. |
| [`14_augment`](https://franckalbinet.github.io/ossl-fuku-transfer/augment.html) | Does pretraining on perturbed spectra help? | No. With perturbations that cover the measured Bruker-Thermo differences, the Kex correlation without fine-tuning is -0.10. |
| [`15_vapour`](https://franckalbinet.github.io/ossl-fuku-transfer/vapour.html) | Does water vapour explain the failure? | No. The Fukushima spectra have about twice the vapour lines of OSSL, but the level is not linked to the Kex errors. |
| [`16_moisture`](https://franckalbinet.github.io/ossl-fuku-transfer/moisture.html) | Does sample moisture explain the failure? | Not alone. Against matched OSSL soils, the Thermo spectra shift in intensity towards 2800 to 3700 cm⁻¹. The Bruker spectra of the same soils barely shift, and the OSSL model fails on them too. |
| [`18_aloh`](https://franckalbinet.github.io/ossl-fuku-transfer/aloh.html) | Does the Al-OH band still track the soil? | The band is not flattened. In Lowland soils, deeper clay-mineral bands go with less clay (Spearman -0.31 and -0.35). In OSSL they go with more clay. |

### Cs-137 ratio

| Notebook | Question | Answer |
|---|---|---|
| [`10_cs137`](https://franckalbinet.github.io/ossl-fuku-transfer/cs137.html) | Do frozen OSSL features predict the Cs-137 ratio? | Yes. On the Cs137 study's test samples, ridge regression on block 4 of the OSSL Kex and CEC models gives R² 0.65, against 0.56 for PLSR on the spectra. These test samples share fields with training. |
| [`11_multitask`](https://franckalbinet.github.io/ossl-fuku-transfer/multitask.html) | Does pretraining on 23 OSSL properties transfer better? | No better than single-task pretraining, for Kex, CEC or the Cs-137 ratio. |
| [`19_cs137_inside`](https://franckalbinet.github.io/ossl-fuku-transfer/cs137_inside.html) | Which spectral features drive the Cs-137 ratio predictions? | The clay-mineral O-H region. The clay-mineral bands relate to the ratio as Cs fixation predicts (Spearman -0.20 in Lowland soils). Clay does not (0.12). |

### Scores on new fields

| Notebook | Question | Answer |
|---|---|---|
| [`20_grouped_cv`](https://franckalbinet.github.io/ossl-fuku-transfer/grouped_cv.html) | How well do the models predict fields they have never seen, by soil type and field type? | R² drops by 0.05 to 0.13. For Kex, OSSL features add nothing. For the Cs-137 ratio they add 0.05 R². Within paddy fields, Kex can hardly be predicted. |

## Questions for the Fukushima data team

1. Fields: does each `field_no` name the same field in every year? `20_grouped_cv` assumes it does.
2. Potassium fertilisation: which fields received extra K after 2011 to limit Cs-137 uptake, and from when? Added K leaves no mineral signature in the spectra. If the paddy fields received it, that would explain why the spectra cannot rank Kex within paddy fields.
3. Kex method: is `soil_ex_K2O` in mg K₂O per 100 g of dry soil, and which extraction gave it? OSSL uses 1 M ammonium acetate at pH 7. The project assumes mg K₂O per 100 g and divides by 47.1 to get cmolc/kg.
4. Other units: are `soil_MgO` and `soil_CaO` in mg of oxide per 100 g of dry soil, and `soil_C` and `soil_N` in %? The comparisons with OSSL in `12_wetchem` depend on these conversions.
5. Clay: which method measured `clay`, and does it differ between soil types or years? If the clay values in the Lowland soils are suspect, the comparisons in `12_wetchem`, `18_aloh` and `19_cs137_inside` need revisiting.
6. Clay mineralogy: which clay minerals dominate the Lowland paddy soils? Do mineralogy data, such as XRD or oxalate-extractable Al and Si, exist for any samples? If the mineralogy is unusual, models trained on Fukushima labels are the way forward rather than OSSL.
7. Sample preparation: how were the samples dried, ground and, if at all, diluted for each instrument? Did the Bruker and Thermo scans of the paired soils use the same prepared material? The answers decide which spectral corrections are realistic.
8. Instruments: which accessory, resolution and purging did the Thermo Nicolet use? Which Bruker instrument and accessory measured the paired soils?
9. Doubtful scans: `00_data` leaves out 12 samples whose scans disagree between operators or between file exports. Which scans are right? The answer can bring these samples back.

## Set-up

The project uses the soilspec packages ([soilspecdl](https://github.com/franckalbinet/soilspecdl), [soilspecdata](https://github.com/franckalbinet/soilspecdata) and [soilspectfm](https://github.com/franckalbinet/soilspectfm)) as editable installs from sibling folders:

```
soilspec/
  ossl-fuku-transfer/
  soilspecdl/
  soilspecdata/
  soilspectfm/
```

Clone the four repositories into one folder, then, in `ossl-fuku-transfer/`:

```sh
uv sync --extra dev
```

The Fukushima data is not public. This repository does not contain it. With access to it, put it in `data/fukushima/` in the workspace folder, beside the project folders, or set `SOILSPEC_DATA` to the folder that holds `fukushima/`. `nbdev-test` skips the cells that read the data. Run `nbdev-test --flags fukudata` to include them.